# 1. Introduction

In the contemporary landscape of urban transportation, car accidents remain a significant concern with far-reaching implications for public safety and road infrastructure. Analyzing the factors contributing to accidents and understanding their severity is crucial for devising effective preventive measures and enhancing road safety standards. In this project, we are going to explore a dataset of over 7M car accidents for some insights, and then build a model to predict severity based on available information.

In [ ]:
from matplotlib import ticker

# We will work with a lot of large numbers
# Which is why I created this function and hid it here

def large_numbers_formatter(decimals=1):
    """Returns a FuncFormatter that formats large numbers in K, M, B notation."""
    def formatter(x, pos):
        if abs(x) >= 1_000_000_000:
            return f"{x / 1_000_000_000:.{decimals}f}B"
        elif abs(x) >= 1_000_000:
            return f"{x / 1_000_000:.{decimals}f}M"
        elif abs(x) >= 1_000:
            return f"{x / 1_000:.{decimals}f}K"
        return f"{x:.{decimals}f}"
    
    return ticker.FuncFormatter(formatter)

In [ ]:
# Importing libraries for data manipulation and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Loading dataset
accidents = pd.read_csv("/kaggle/input/us-accidents/US_Accidents_March23.csv", parse_dates=['Start_Time', 'End_Time'])

accidents.head()

# 2. Data Exploration and Preprocessing:

Before going to the main part of our analysis, let's check the main features of the dataset and also ensure data quality. The dataset has a big range of attributes: severity, weather conditions, road features, and temporal characteristics. For now, we will focus on missing values, outliers, and inconsistencies.

In [ ]:
# Display main data characteristics
round(accidents.describe())

* From this information we can see that there are some missing values in a few columns. We'll deal with them while preprocssing the data for the ML model;
* We can use some of the columns to derive new useful information about the car crashes;
* It's good to see that there are no negative values where they shouldn't exist (for example, negative precipitation), which means we don't have to hunt for very weird data.

In [ ]:
# First, let's check the amount of missing variables:
share_missing = round(accidents.isna().mean().sort_values(ascending=False), 4) * 100
display(share_missing[share_missing != 0])

For most columns it seems we just don't have info in a negligible amount of cases, but: 
* The end place could be missing because the distance is just not that significant
* precipitation and wind chill could be missing because the values are zero.

Let's check the values in these cases below and fill them or adapt accordingly.

## Missing ending positions

In [ ]:
# Checking if all missing latitudes have a corresponding missing longitude
display(accidents[accidents['End_Lat'].isna()]['End_Lng'].value_counts())

Seems like there's no case where the lat is missing, but the longitude is not (and vice versa), this is good, as having something like that would imply trickier errors in the data.

In [ ]:
# Checking the median distance for rows where end latitude is missing
accidents['Missing End_Lat'] = accidents['End_Lat'].isna()
display(accidents.groupby('Missing End_Lat')['Distance(mi)'].median())

We have established that for cases where there is no ending position, the median distance is zero, so indeed the missing values seem related to small distances. Nevertheless, there are a few outliers: the maximum distance for missing positions is 441 miles. this seems unreal.

In [ ]:
# Setting style for my charts

sns.set(style="dark", 
        palette="deep", 
        context="notebook",)

plt.figure(figsize=(15,5))

sns.histplot(x='Distance(mi)', 
             data=accidents[accidents['Distance(mi)'] >= 10], 
             hue='Missing End_Lat', 
             log_scale=(True, False),
             element='step')
plt.gca().xaxis.set_major_formatter(large_numbers_formatter(0))

plt.title('Distribution of Distances Over 10 Miles According to Missing/Present Latitudes')
plt.show()

There are a few thousands of accidents with missing end coordinates and over 10 miles distance. While the number might be big, the dataset has over 7 million rows, and these cases seem quite extreme outliers. Let's see if their description shows any clues of what is happening.

In [ ]:
accidents[accidents['Distance(mi)'] >= 10].sort_values('Distance(mi)', ascending=False)['Description'][:10].to_list()

*Well*, I am not an expert in US highways, but normally accidents that involve blocking two lanes or both ways are usually very severe and can cause kilometer-long traffic jams in busy highways. Could it get over 100 miles distance? I don't know, as I speak kilometers, but also from the field description it's not possible to say whether the distance is the total distance affected by the accident (including traffic jams), or if the distance should be only the distance from the beginning to the end of the accident area. Anyway, I am still not sure if I should leave car accidents with over 10 miles distance in the dataset, but let's try to feature engineer these two conditions from the description.

In [ ]:
accidents['Multiple Lanes Closed'] = accidents['Description'].str.contains('lanes.*blocked|lanes.*closed|lanes.*obstructed').astype('boolean')
accidents['Both Ways Affected'] = accidents['Description'].str.contains('both.*ways').astype('boolean')

fig, (ax1, ax2) = plt.subplots(2,1,figsize=(15,8))

for ax, hue in zip([ax1,ax2],['Multiple Lanes Closed', 'Both Ways Affected']):
    sns.histplot(x='Distance(mi)',
                 data=accidents[accidents['Distance(mi)'] >= 10],
                 hue=hue,
                 log_scale=(True, False),
                 element='step',
                ax=ax)
    ax.get_xaxis().set_major_formatter(large_numbers_formatter(0))

plt.suptitle('Distribution of Distances Over 10 Miles According to New Features')
plt.show()

I was expecting these feature to become more frequent as distances increase, but apparently not. Anyway, they might be useful in the second part of this notebook, where I'll try to predict severity.

## Column Types

In [ ]:
# Checking data types
accidents.info()

Nothing in this long list of columns draws my attention right now, so let's move on to the EDA part of the project.

# 3. Exploratory Data Analysis (EDA):

Now we understand better the missing values and the data types of the dataset, we can move on to the EDA part and create a few visualizations:

* Temporal Patterns and Severity Analysis

We will make a detailed breakdown of accident events by hour. But our focus goes beyond mere representation. Our visualizations will show how severe the accidents are and how often they happen. This picture will help us see how things change during the 24-h period and how they affect accident frequency and severity.

* Correlation of Severity with Roadway Attributes

A crucial aspect of our analysis lies in the examination of how diverse road features influence accident severity. I will use a correlation chart that show how accidents are related to different road characteristics. These insights might offer a deeper understanding of the variables that may exacerbate or mitigate accident severity.

* Spatial Insight through Heatmapping

I'll make a spatial analysis is in the form of a geographically oriented heatmap, which shows where accidents are more likely to happen. This visualization leverages a gradient spectrum to portray the density of accidents, enabling us to pinpoint specific regions that demand closer attention. 

## Temporal Breakdown

In [ ]:
from matplotlib.ticker import PercentFormatter

# Changing Severity to Category for Visualizations
accidents['Severity'] = pd.Categorical(accidents['Severity'], categories=[1,2,3,4], ordered=True)

# Extracting hour of day from 'Start_Time'
accidents['Start Hour'] = accidents['Start_Time'].dt.hour

# Charts for visualizing Frequencies per hour
fig, (ax1, ax2) = plt.subplots(1,2, figsize=(15,7))

hour_counts = accidents.pivot_table(index='Start Hour', columns='Severity', values='ID', aggfunc='count')
hour_counts.plot(kind='line', marker='o', ax=ax1)

ax1.get_yaxis().set_major_formatter(large_numbers_formatter(0))
ax1.set_title('Accident Frequency by Hour')

hour_incidence = hour_counts.div(hour_counts.sum(axis=1), axis=0)
hour_incidence.plot(kind='line', marker='o', ax=ax2)

ax2.set_title('Acccident Severity Ratio by Hour')
ax2.get_yaxis().set_major_formatter(PercentFormatter(xmax=1.0))
                                  
plt.suptitle('Accident Frequency & Severity Ratio by Hour')
plt.show()

* Accidents are clearly affected by the time of day: As expected, most accidents occur during the commute from home to work or from work to home;
* But when we check the Severity Ratio, there's also a clear increase in the frequency of highest-severity accidents from 23h to 4h;

Since we started talking about time, let's get a few more insights from this data by extracting days of the week and holidays.

In [ ]:
import holidays # Testing the holidays library for the first time!

# Creating a mapping for the days of the week
day_mapping = {
    0: 'Monday',
    1: 'Tuesday',
    2: 'Wednesday',
    3: 'Thursday',
    4: 'Friday',
    5: 'Saturday',
    6: 'Sunday'
}

# Getting temporal data from the start time column
accidents['Day of Week'] = accidents['Start_Time'].dt.day_of_week
accidents['Day of Week'] = accidents['Day of Week'].map(day_mapping)
accidents['Day of Week'] = pd.Categorical(accidents['Day of Week'], categories=day_mapping.values(), ordered=True)

# Getting US holidays
us_holidays = holidays.US(years=range(accidents['Start_Time'].dt.year.min(), accidents['Start_Time'].dt.year.max()+1))

# Creating a Boolean column
accidents['Holiday'] = accidents['Start_Time'].dt.date.astype('datetime64').isin(us_holidays)

fig, (ax1, ax2) = plt.subplots(1,2,figsize=(15,7))

# Using pivots for faster plotting (7M rows folks)
dow_pivot = accidents.pivot_table(index='Day of Week', columns='Severity', values='ID', aggfunc='count')
dow_pivot = dow_pivot.div(dow_pivot.sum(axis=0), axis=1)
dow_pivot.plot(ax=ax1, marker='o')

ax1.set_title('Acccident Distribution Across Days of the Week')
ax1.get_yaxis().set_major_formatter(PercentFormatter(xmax=1.0))

holidays_pivot = accidents.pivot_table(index='Severity', columns='Holiday', values='ID', aggfunc='count')
holidays_pivot = holidays_pivot.div(holidays_pivot.sum(axis=0), axis=1)
holidays_pivot.plot(ax=ax2, kind='bar')

ax2.set_title('Share of Accident Severity on Holidays vs Normal Days')
ax2.get_yaxis().set_major_formatter(PercentFormatter(xmax=1.0))
ax2.tick_params('x', rotation=0)
plt.show()

* Low-severity accidents are much more likely to occur during weekdays than weekends, whereas that distribution is more balanced for high-severity accidents;
* Holidays also seem to have a higher share of accidents with severity 2 and 4 than normal days;

In [ ]:
fig, ((ax1,ax2,ax3),(ax4,ax5,ax6)) = plt.subplots(2,3,figsize=(15,8), sharex=True)

# Chosen metrics for the chart below
metrics_list = ['Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)']

# Creating a super chart with multiple numerical features
for ax, metric in zip([ax1,ax2,ax3,ax4,ax5,ax6], metrics_list):
    metric_pivot = accidents.pivot_table(index='Severity', values=metric, aggfunc='median')
    metric_pivot.plot(kind='bar', ax=ax, legend=False)
    ax.set_title(metric)
    ax.tick_params('x',rotation=0)

plt.suptitle('Median Severity Across Temperature, Wind Chill, Humidity, Pressure, Visibility, and Wind Speed')
plt.show()

* Lower median temperature and wind chill clearly contribute to higher-severity accidents, likely;
* Higher humidity is also associated with higher severity;
* The relation between wind speed and severity is less straightforward, with higher wind speed being associated with severity 3, but not with severity 4;
* Other metrics have some small effects on severity.

## Calculating Accident-Prone Places
We already checked features of the dataset related to time and dates, then moved on to see how our numerical features vary with severity. Now let's check our boolean columns, which are mostly references to specific places of roads.

In [ ]:
# Selecting boolean columns for heatmap
booleans = accidents.select_dtypes('boolean').columns.to_list()[:-4]

# Calculating the number of accidents for each combination of severity and boolean columns
severity_bools = accidents.groupby('Severity')[booleans].sum()

# Creating a heatmap to visualize the relationship between accident severity and boolean columns
plt.figure(figsize=(15, 5))

# Formatting numbers for heatmap
formatter = large_numbers_formatter(0)
formatted_annot = severity_bools.applymap(lambda x: formatter(x, None))

# heatmapping
heatmap = sns.heatmap(severity_bools, annot=formatted_annot, cmap='viridis', fmt='')

# Getting the colorbar from the heatmap object
cbar = heatmap.collections[0].colorbar

# Applying formatter to the colorbar
cbar.ax.yaxis.set_major_formatter(large_numbers_formatter(1))

plt.title('Accidents per Road Features')
plt.xlabel('Road Features')
plt.ylabel('Severity')
plt.tight_layout()

plt.show()

From our heatmap, we can see that:

* Junctions, crossings and traffic signals tend to have most car accidents;
* Junctions and traffic signals tend to have the most severe accidents;
* Roundabouts and Turning Loops have the least accidents, although Turning Loops are suspiciously low and we might want to check this column later;

## Visualizing States According to Ratio of Severe Accidents

Let's also have a look on the accident data

In [ ]:
# Calculating data by state
most_accidents = accidents.pivot_table(index='State', columns='Severity', values='ID',  aggfunc='count').sort_values(2, ascending=False)
accident_ratio = most_accidents.div(most_accidents.sum(axis=1), axis=0).dropna(how='all')

# Creating plots
fig, (ax1, ax2) = plt.subplots(2,1,figsize=(15,7),sharex=True)

# Absolute number of accidents
most_accidents.plot(kind='bar', stacked=True, ax=ax1)
ax1.get_yaxis().set_major_formatter(large_numbers_formatter(1))

# Share of severe accidents
accident_ratio.plot(kind='bar', stacked=True, ax=ax2)
ax2.get_yaxis().set_major_formatter(PercentFormatter(1))

plt.suptitle('Most Accidents vs Share of Accidents by Severity')
plt.show()

* California, Florida and Texas are the states with the most accidents. These are all states with a large population, and are bound to have more accidents;
* South Dakota, Wyoming and Arkansas are the states with the highest share of severe accidents. South Dakota has the least accidents in our chart, so even a high share of severe accidents is probably a very small number;

## Folium Map Visualization

* I should note that this heatmap is not scaled to account for traffic levels in these areas, which means it does not show the rate of accidents relative to traffic, but simply the absolute number of accidents.
* Also worth noting that the data is limited to a few states, as seen in the map below.

In [ ]:
import folium
from folium.plugins import HeatMap

# Creating a folium map centered at a specific location
m = folium.Map(location=[accidents['Start_Lat'].mean(), accidents['Start_Lng'].mean()], zoom_start=4.4)

# Converting data to a list of coordinates
# It was necessary to reduce the number of rows to avoid crashing the notebook
data = accidents.sort_values('End_Time')[['Start_Lat', 'Start_Lng']][:1000000].values.tolist()

# Creating a heatmap layer and adding it to the folium map
HeatMap(data).add_to(m)

# Displaying
m

# 4. Feature Extraction & Engineering:

Next, we will be looking at the data to find hidden insights. Here is a brief overview of our manipulations and why they matter:

* Time Insights: We already extracted hours, days of the week and holidays. Now let's also get month quarters. This will help with some categorical data to spot trends linked to different times, seasonal variations.

* Duration Breakdown: Calculating accident duration and converting it to minutes helps us understand how the duration impacts the severity.

* Smart Categories: Let's create categories and order them where needed. This categorization streamlines data for our models, ensuring they grasp the context and make smarter predictions.

* Trimming Unnecessary Columns: We have multiple columns with 'Day/Night' variations. This might actually make our model less precise and needs to be dealt with.

These transformations will be really useful for improving predictive power of our model.

## Multiple Day & Night Columns
We don't need this many columns with repetitive labels as they are somewhat redundant. Let's instead create one column with categories like 'Day', 'Twilight' and 'Night' and drop the rest. 

After some research, it seems that the most disagreement would happen between the 'Sunrise_Sunset' column and the 'Astronomical Twilight' column, so we will use them as base for our 'twilight'category.

In [ ]:
# Setting conditions
conditions = [(accidents['Sunrise_Sunset'] == 'Day'),
              (accidents['Sunrise_Sunset'] == 'Night') & (accidents['Astronomical_Twilight'] == 'Day')]
# Creating columns
accidents['Sunlight'] = np.select(conditions, ['Day', 'Twilight'], default='Night')

## Deriving Additional Time & Date Columns
Since seasonal trends can play a role in Accident severity, let's derive months and quarters from the start time.
Duration can also be a important factor of severity, so we will get the accident duration in minutes.

In [ ]:
# Extracting time-related features
accidents['Month'] = accidents['Start_Time'].dt.month
accidents['Quarter'] = accidents['Start_Time'].dt.quarter
accidents['Duration (mins)'] = (accidents['End_Time'] - accidents['Start_Time']).dt.total_seconds() / 60

# Creating Categories
categories = ['Sunlight', 
              'Weather_Condition', 
              'Wind_Direction', 
              'Month', 
              'Quarter', 
              'Day of Week']

accidents[categories] = accidents[categories].astype('category')

# 5. Preprocessing & Predictive Modeling:

## Preprocessing

During the preprocessing phase, we need to impute missing values in numeric features using the mean of each respective column, ensuring that the subsequent analysis remained uninfluenced by incomplete records. For categorical attributes we can use an 'unknown' category to handle absent data.

We can also standardize & scale numerical features, aiding in the optimization of modeling performance. Categorical variables should be encoded using a one-hot encoding technique, enabling the incorporation of non-ordinal information into the analysis.

The resulting cleaned and transformed dataset can then be used for our model. 

### Is target Encoding Valid For this Dataset?

Target encoding can very easily lead to data leakage, but in this dataset it does make sense to use it, considering the following:
* Accidents can be location-specific (for example, particular junctions, streets, and traffic signals can be specially susceptible to car accidents), which means a location with a history of car accidents would indeed be more likely to have such accidents in the future;
* We can use specific encoding to hedge against overfitting;
* We can train and test the data in multiple splits to prevent the target encoding from skewing results.

## Predictive Modeling

Having meticulously prepared our dataset, we proceeded to construct a predictive model using the XGBoost algorithm. XGBoost is known for its efficiency and accuracy, enabling us to unravel the intricate relationships between the myriad of features and the target variable—accident severity.

We partitioned the preprocessed dataset into distinct training and testing sets to ensure the model's ability to generalize. Since we have a multi-class classification task, we need to set the model's objective to 'multi:softmax,' accommodating the grading of severity levels spanning 1 to 4.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from category_encoders import MEstimateEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier

# I tried this and the performance got worse, so it's commented out for now
#sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

# Creating X and y from original dataset & dropping unnecessary columns
X = accidents.copy().drop(columns=['Start_Time', 'End_Time', 'Description', 'ID', 'Source', 'Timezone', 
                                   'Airport_Code', 'Weather_Timestamp', 'Sunrise_Sunset', 
                                   'Civil_Twilight', 'Nautical_Twilight', 'Astronomical_Twilight',
                                  'Country'])
y = X.pop('Severity')
y = y.astype('int') -1
# Splitting the data into train and test sets for both X and y
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42)

# Columns to include in the pipeline
numeric_cols = X.select_dtypes('number').columns.to_list()
categorical_cols = categories

# Columns to encode with MEstimate
target_encode_cols = ['Street', 'City', 'State', 'Zipcode', 
                      'Wind_Direction', 'Day of Week', 'Month', 
                      'Quarter', 'Weather_Condition', 'Sunlight']

# Numeric columns
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# Target-encoded categorical columns
target_encoder = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('m_estimate', MEstimateEncoder(m=2))
])

# Preprocessor combining everything
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_cols),
    ('target_cat', target_encoder, target_encode_cols)
])

# Creating the model
xgb_model = XGBClassifier(
    objective='multi:softmax',
    num_class=4,
    random_state=42,
    tree_method='gpu_hist'
)

# Creating the pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', xgb_model)
])

# Fitting the pipeline on the training data
pipeline.fit(X_train, y_train)

# Predicting
predictions = pipeline.predict(X_val)

## Model Results: Confusion Matrix

Below we'll show a Confusion Matrix detailing how the model performed against the real data. Ideally, we would have a perfect bright diagonal, but as you can see below, the model misclassified many of the accidents as having Severity 2. This is not surprising given that the vast majority of accidents are in this category, and is something we can try to improve in the next steps.

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

# Compute confusion matrix
conf_mat = confusion_matrix(y_val, predictions)

# Create a heatmap for the confusion matrix
plt.figure(figsize=(8, 6))
matrix_heatmap = sns.heatmap(conf_mat, annot=True, fmt='d', cmap='viridis', xticklabels=['1', '2', '3', '4'], yticklabels=['1', '2', '3', '4'])

# Getting the colorbar from the heatmap object
matrix_cbar = matrix_heatmap.collections[0].colorbar

# Applying formatter to the colorbar
matrix_cbar.ax.yaxis.set_major_formatter(large_numbers_formatter(1))

plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

class_report = classification_report(y_val, predictions, target_names=['1', '2', '3', '4'])
print(class_report)

## Visualizing Distribution of Severity Levels

As we have seen in the Confusion matrix, there are some issues with severity levels 1 and 4 as they are much less common. Let's create another visualization of the predicted vs actual models.

In [ ]:
# Creating subplots
fig, (ax1,ax2) = plt.subplots(1, 2, figsize=(15, 5), sharey=True)

# Creating a list of chart parameters
plots = [[predictions, 'Predicted'], [y_val, 'Actual']]

# Two plots for the price of one
for ax, plot in zip([ax1,ax2], plots):
    sns.histplot(plot[0], bins=[0,1,2,3,4], ax=ax)
    ax.set_xlabel(f'{plot[1]} Severity')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Distribution of {plot[1]} Severity Levels')
    ax.set_xticks([1,2,3,4])
    ax.set_xticklabels([1,2,3,4], ha='left')
    ax.get_yaxis().set_major_formatter(large_numbers_formatter(1))

plt.suptitle('Actual vs Predicted Severity')
plt.show()

## Visualizing Feature Importances

The model importances help us understand what the models consider to be the most influential data for classifying the severity. There are 

In [ ]:
# Get transformed feature names from numeric part
numeric_features = numeric_cols  # already just names

# Get transformed feature names from target encoding
# category_encoders doesn't support get_feature_names_out by default,
# so we have to manually grab the original column names
categorical_features = target_encode_cols

# Combine into one full list in the right order
feature_names = numeric_features + categorical_features

# Get feature importances from the XGB model
importances = pipeline.named_steps['model'].feature_importances_

# Combine into DataFrame
importances_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

fig, (ax1,ax2) = plt.subplots(1,2,figsize=(15,5))

# Most influential features
sns.barplot(data=importances_df[:10], x='Importance', y='Feature', ax=ax1)
ax1.set_title('Top 10 Most Influential Features')

# Least influential features
sns.barplot(data=importances_df[-10:], x='Importance', y='Feature', ax=ax2)
ax2.set_title('Top 10 Least Influential Features')

plt.suptitle('Feature Importances from Model')
plt.tight_layout()
plt.show()

# 6. Cross-Validation and Fine-Tuning:

So, the results of our first run were good, but not *ideal*. Below we will try to optimize the model with GridSearchCV, but due to the amount of data, we will be using only 5% of it for now (that's about 140K accidents). After all, in this stage, we are just after a benchmark to compare how different model parameters perform. Once we have a champion, we will try running it again on the full data.

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

# Creating samples with only 10% of the dataset
X_sample, _, y_sample, _ = train_test_split(
    X, y, train_size=0.05, random_state=42, stratify=y
)

# Creating a parameter grid for tuning
param_grid = {
    'model__eta': [0.01, 0.05],
    'model__subsample': [0.5, 1],
    'model__colsample_bytree': [0.7, 0.8],
    'model__reg_alpha': [0.1, 0.5],
    'model__reg_lambda': [0.8, 1.0],
    'model__gamma':[0, 0.5]
}

# Base model with fixed params
base_params = {
    'objective': 'multi:softmax',
    'num_class': 4,
    'eval_metric': 'mlogloss',
    'tree_method': 'gpu_hist',
    'use_label_encoder': False
}

# Creating the pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(**base_params))
])

# Grid searching with cross-validation
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=3,  # 3-fold CV to reduce compute
    scoring='f1_macro',
    n_jobs=-1
)

# Running the search
grid_search.fit(X_sample, y_sample)

print("Best parameters found:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

In [ ]:
# Setting new hyperparameters
params = {
    'objective': 'multi:softmax',
    'num_class': 4, 
    'eval_metric': 'mlogloss', 
    'eta': 0.05,  
    'subsample': 1,  
    'colsample_bytree': 0.8, 
    'gamma': 0.5,  
    'reg_alpha': 0.1,  
    'reg_lambda': 0.8,  
    'tree_method': 'gpu_hist'  # Enable GPU acceleration
}

# Creating a new XGBoost model
new_model = XGBClassifier(**params)

# Create the pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', new_model)
])

# Fit the pipeline to the training data
pipeline.fit(X_train, y_train)

# Make predictions on the validation set
new_predictions = pipeline.predict(X_val)

# Compute confusion matrix
conf_mat = confusion_matrix(y_val, predictions)

# Create a heatmap for the confusion matrix
plt.figure(figsize=(8, 6))
matrix_heatmap = sns.heatmap(conf_mat, annot=True, fmt='d', cmap='viridis', xticklabels=['1', '2', '3', '4'], yticklabels=['1', '2', '3', '4'])

# Getting the colorbar from the heatmap object
matrix_cbar = matrix_heatmap.collections[0].colorbar

# Applying formatter to the colorbar
matrix_cbar.ax.yaxis.set_major_formatter(large_numbers_formatter(1))

plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

# Evaluate the pipeline
print(classification_report(y_val, new_predictions))

In [ ]:
# Creating subplots
fig, (ax1,ax2) = plt.subplots(1, 2, figsize=(15, 5), sharey=True)

# Creating a list of chart parameters
plots = [[new_predictions, 'Predicted'], [y_val, 'Actual']]

# Two plots for the price of one
for ax, plot in zip([ax1,ax2], plots):
    sns.histplot(plot[0], bins=[0,1,2,3,4], ax=ax)
    ax.set_xlabel(f'{plot[1]} Severity')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Distribution of {plot[1]} Severity Levels')
    ax.set_xticks([1,2,3,4])
    ax.set_xticklabels([1,2,3,4])
    ax.get_yaxis().set_major_formatter(large_numbers_formatter(1))

plt.suptitle('Actual vs Predicted Severity')
plt.show()

# 7. Performance Evaluation & Conclusions

After comparing both models, it seems that we only got marginal improvements. This is normal, as due to the size of the dataset and the computing power available, we had to opt for some time-saving options in favor of a more thorough but compute-heavy approach. In any case, we:
* Explored the missing data in the dataset and understood the main reasons behind it;
* Dived in the data with visualizations to better understand how time affects the severity of car crashes;
* Engineered a few features based on these new insights;
* Created a model that can predict the severity of car crashes relatively well, explored its performance and features;
* And conducted some fine-tuning based on model sampling;

Thanks a lot for reading all the way here! If you liked this project or have comments / feedback on improvements, let me know!